In [ ]:
    # ============================================================
    # CATA — ADAPTIVE AUGMENTATION
    # ============================================================

    # Стратегия 0: После правки(плохая - не симметричное распределение от среднего)

    # def _apply_augmentation(self, x_num, x_cat, importance=None):
    #     """
    #     Computes per-feature masking probabilities and generates masks.
    #     Masking is applied at embedding level in forward_contrastive.

    #     CATA mode (CATA_alpha > 0):
    #         P_mask(i) = clamp(CR * (1 - alpha * importance(i)), 0.05, 0.95)
    #     Random mode (CATA_alpha = 0):
    #         P_mask(i) = CR (uniform)

    #     Args:
    #         x_num: [batch, n_num_features]
    #         x_cat: [batch, n_cat_features]
    #         importance: [batch, n_tokens] from CATA, ignored if CATA_alpha=0
    #     Returns:
    #         x_num, x_cat: unchanged input data
    #         mask_num: [batch, n_num_features] boolean mask
    #         mask_cat: [batch, n_cat_features] boolean mask
    #     """
    #     if x_num is not None:
    #         batch_size = x_num.shape[0]
    #         device = x_num.device
    #     elif x_cat is not None:
    #         batch_size = x_cat.shape[0]
    #         device = x_cat.device
    #     else:
    #         return None, None, None, None

    #     mask_num = None
    #     if x_num is not None and self.n_num_features > 0:
    #         if self.CATA_alpha and importance is not None:
    #             # Adaptive masking based on feature importance
    #             prob_num = self.corruption_rate * (1 - self.CATA_alpha * importance[:, :self.n_num_features])
    #             prob_num = torch.clamp(prob_num, 0.05, 0.95)
    #         else:
    #             # Random masking (ablation mode)
    #             prob_num = torch.full((batch_size, self.n_num_features), self.corruption_rate, device=device)

    #         mask_num = torch.rand(batch_size, self.n_num_features, device=device) < prob_num

    #         # Ensure at least one feature is masked
    #         if x_cat is None or self.n_cat_features == 0:
    #             mask_num |= ~mask_num.any(dim=1, keepdim=True)

    #     mask_cat = None
    #     if x_cat is not None and self.n_cat_features > 0:
    #         if self.CATA_alpha and importance is not None:
    #             prob_cat = self.corruption_rate * (1 - self.CATA_alpha * importance[:, self.n_num_features:])
    #             prob_cat = torch.clamp(prob_cat, 0.05, 0.95)
    #         else:
    #             prob_cat = torch.full((batch_size, self.n_cat_features), self.corruption_rate, device=device)

    #         mask_cat = torch.rand(batch_size, self.n_cat_features, device=device) < prob_cat

    #         if x_num is None or self.n_num_features == 0:
    #             mask_cat |= ~mask_cat.any(dim=1, keepdim=True)

    #     return x_num, x_cat, mask_num, mask_cat





    # Стратегия 1: ПРИМЕНЯЕМ ДЛЯ CATA - симметричное распределение при котором ВАЖНЫЕ признаки маскируются ЧАЩЕ чем неважные - ИННОВАЦИОННЫЙ ПОДХОД
    def _apply_augmentation(self, x_num, x_cat, importance=None):
        """
        Computes per-feature masking probabilities and generates masks.

        NEW: INVERTED CATA - important features are masked MORE often.
        Formula: P_mask(i) = clamp(CR * (1 + alpha * (importance(i) - 1)), 0.05, 0.95)

        Args:
            x_num: [batch, n_num_features]
            x_cat: [batch, n_cat_features]
            importance: [batch, n_tokens] from CATA, ignored if CATA_alpha=0
        Returns:
            x_num, x_cat: unchanged input data
            mask_num: [batch, n_num_features] boolean mask
            mask_cat: [batch, n_cat_features] boolean mask
        """
        if x_num is not None:
            batch_size = x_num.shape[0]
            device = x_num.device
        elif x_cat is not None:
            batch_size = x_cat.shape[0]
            device = x_cat.device
        else:
            return None, None, None, None

        mask_num = None
        if x_num is not None and self.n_num_features > 0:
            if self.CATA_alpha > 0 and importance is not None:
                # ============================================================
                # INVERTED CATA: важные признаки маскируются ЧАЩЕ!
                # ============================================================
                # importance имеет mean=1
                # importance=2.0 → P = CR * (1 + alpha * (2-1)) = CR * (1 + alpha)
                # importance=0.5 → P = CR * (1 + alpha * (0.5-1)) = CR * (1 - alpha*0.5)
                # importance=1.0 → P = CR (базовая линия)
                # ============================================================
                imp = importance[:, :self.n_num_features]
                prob_num = self.corruption_rate * (1 + self.CATA_alpha * (imp - 1))
                prob_num = torch.clamp(prob_num, 0.05, 0.95)
            else:
                prob_num = torch.full((batch_size, self.n_num_features), self.corruption_rate, device=device)

            mask_num = torch.rand(batch_size, self.n_num_features, device=device) < prob_num

            # Гарантируем хотя бы одну маску
            if x_cat is None or self.n_cat_features == 0:
                mask_num |= ~mask_num.any(dim=1, keepdim=True)

        mask_cat = None
        if x_cat is not None and self.n_cat_features > 0:
            if self.CATA_alpha > 0 and importance is not None:
                # ============================================================
                # INVERTED CATA для категориальных признаков
                # ============================================================
                imp = importance[:, self.n_num_features:]
                prob_cat = self.corruption_rate * (1 + self.CATA_alpha * (imp - 1))
                prob_cat = torch.clamp(prob_cat, 0.05, 0.95)
            else:
                prob_cat = torch.full((batch_size, self.n_cat_features), self.corruption_rate, device=device)

            mask_cat = torch.rand(batch_size, self.n_cat_features, device=device) < prob_cat

            if x_num is None or self.n_num_features == 0:
                mask_cat |= ~mask_cat.any(dim=1, keepdim=True)

        return x_num, x_cat, mask_num, mask_cat




    # Стратегия 2: ДЛЯ АБЛЯЦИИ CATA - симметричное распределение при котором НЕважные признаки маскируются ЧАЩЕ чем неважные - КЛАССИЧЕСКИЙ ПОДХОД
    def _apply_augmentation(self, x_num, x_cat, importance=None):
        """
        SYMMETRIC CATA: средний = CR, важные → реже, неважные → чаще
        Formula: P_mask(i) = clamp(CR * (1 - alpha * (importance(i) - 1)), 0.05, 0.95)
        """
        if x_num is not None:
            batch_size = x_num.shape[0]
            device = x_num.device
        elif x_cat is not None:
            batch_size = x_cat.shape[0]
            device = x_cat.device
        else:
            return None, None, None, None

        mask_num = None
        if x_num is not None and self.n_num_features > 0:
            if self.CATA_alpha > 0 and importance is not None:
                # ============================================================
                # SYMMETRIC CATA: средний = CR, важные → реже, неважные → чаще
                # ============================================================
                imp = importance[:, :self.n_num_features]
                prob_num = self.corruption_rate * (1 - self.CATA_alpha * (imp - 1))
                prob_num = torch.clamp(prob_num, 0.05, 0.95)
            else:
                prob_num = torch.full((batch_size, self.n_num_features), self.corruption_rate, device=device)

            mask_num = torch.rand(batch_size, self.n_num_features, device=device) < prob_num

            if x_cat is None or self.n_cat_features == 0:
                if not mask_num.any(dim=1).all():
                    for i in range(batch_size):
                        if not mask_num[i].any():
                            mask_num[i, torch.randint(0, self.n_num_features, (1,))] = True

        mask_cat = None
        if x_cat is not None and self.n_cat_features > 0:
            if self.CATA_alpha > 0 and importance is not None:
                imp = importance[:, self.n_num_features:]
                prob_cat = self.corruption_rate * (1 - self.CATA_alpha * (imp - 1))
                prob_cat = torch.clamp(prob_cat, 0.05, 0.95)
            else:
                prob_cat = torch.full((batch_size, self.n_cat_features), self.corruption_rate, device=device)

            mask_cat = torch.rand(batch_size, self.n_cat_features, device=device) < prob_cat

            if x_num is None or self.n_num_features == 0:
                if not mask_cat.any(dim=1).all():
                    for i in range(batch_size):
                        if not mask_cat[i].any():
                            mask_cat[i, torch.randint(0, self.n_cat_features, (1,))] = True

        return x_num, x_cat, mask_num, mask_cat